# A recurrent neural network

An RNN processes a sequence one step at a time, carrying a hidden state that
summarises everything seen so far:

```
state_t  = activation(U x_t + W h_(t-1))
output_t = V state_t
```

The same three weight matrices are reused at every step. That weight sharing
across time is what makes it *recurrent*, and it is why an RNN can handle a
sequence of any length with a fixed number of parameters.

The task here is next-number prediction: given a run of consecutive integers,
predict the one that follows at every position.


In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from si.data import Dataset
from si.util import train_test_split, multiclass_accuracy
from si.supervised.nn import NN, Adam, RNN, SoftMax

## The data

Each example is 10 consecutive integers starting somewhere in 0-9, one-hot
encoded over 20 possible values. The target at each step is the *next* number,
so the model must learn "add one" from the data alone.

Tensors are `(examples, timesteps, features)`, the same layout the attention
layers use in eval8.


In [ ]:
def to_categorical(x, n_col=None):
    """ One-hot encoding of nominal values """
    if not n_col:
        n_col = np.amax(x) + 1
    one_hot = np.zeros((x.shape[0], n_col))
    one_hot[np.arange(x.shape[0]), x] = 1
    return one_hot

def gen_num_seq(nums, seq_len=10, n_col=20):
    """ Sequences of consecutive integers, with the next one as the target """
    X = np.zeros([nums, seq_len, n_col], dtype=float)
    y = np.zeros([nums, seq_len, n_col], dtype=float)
    for i in range(nums):
        start = np.random.randint(0, 10)
        # the inputs are start .. start+9, the targets start+1 .. start+10,
        # so every position's target is genuinely the NEXT number -- rolling
        # the inputs instead would wrap the first value round to the end and
        # ask the model to predict something unknowable.
        X[i] = to_categorical(np.arange(start, start + seq_len), n_col=n_col)
        y[i] = to_categorical(np.arange(start + 1, start + seq_len + 1), n_col=n_col)
    return Dataset(X, y)

In [ ]:
np.random.seed(0)
dataset = gen_num_seq(3000)
print('X:', dataset.X.shape, ' y:', dataset.y.shape)
print()
# Read a couple of examples back as numbers rather than one-hot rows.
for i in range(3):
    print('input ', dataset.X[i].argmax(axis=-1).tolist())
    print('target', dataset.y[i].argmax(axis=-1).tolist())
    print()
print('every target row is a single one-hot:',
      bool(np.all(dataset.y.sum(axis=-1) == 1)))

In [ ]:
train, test = train_test_split(dataset, split=0.8)
print(f'{len(train)} train, {len(test)} test')

## The model

One RNN layer with 10 hidden units, then a softmax over the 20 possible
values. `bptt_trunc=5` truncates backpropagation through time to five steps:
the gradient is carried back five steps rather than the full ten, which bounds
the cost and limits exploding gradients — at the price of not learning
dependencies longer than the window.


In [ ]:
np.random.seed(1)
model = NN(optimizer=Adam(), epochs=60, verbose=True, step=10)
model.add(RNN(10, bptt_trunc=5, input_shape=(10, 20)))
model.add(SoftMax())
print(model)

## Training

Sixty epochs is enough here — a couple of seconds. (Earlier versions of this
notebook ran 20000, which is many minutes for no additional benefit; the plot
below shows where the loss actually flattens.)


In [ ]:
model.fit(train)

In [ ]:
losses = [model.history[e][0] for e in sorted(model.history)]

plt.figure(figsize=(6, 3.5))
plt.plot(range(1, len(losses) + 1), losses)
plt.xlabel('epoch'); plt.ylabel('MSE loss'); plt.yscale('log')
plt.title('training loss'); plt.tight_layout(); plt.show()

print(f'loss {losses[0]:.4f} -> {losses[-1]:.4f}')

## Evaluation

The point of training is a model that works on data it has not seen.
Accuracy here is per POSITION: each of the 10 steps in every test sequence
is a separate prediction, so a sequence only counts as fully correct if all
ten are.


In [ ]:
predictions = model.predict(test.X)
print('predictions:', predictions.shape)
print()
print(f'per-position accuracy on the test set: '
      f'{multiclass_accuracy(test.y, predictions):.4f}')

predicted_ids = predictions.argmax(axis=-1)
target_ids = test.y.argmax(axis=-1)
whole = (predicted_ids == target_ids).all(axis=1).mean()
print(f'fully correct sequences:                {whole:.4f}')

In [ ]:
# What it actually predicts, as numbers.
for i in range(5):
    given = test.X[i].argmax(axis=-1)
    want = target_ids[i]
    got = predicted_ids[i]
    mark = 'ok ' if (want == got).all() else 'ERR'
    print(f'{mark} input      {given.tolist()}')
    print(f'    predicted  {got.tolist()}')
    print(f'    target     {want.tolist()}')
    print()

In [ ]:
# Accuracy by timestep. It is flat at 1.0, including position 0 -- which is
# worth pausing on, because it says something about the TASK rather than the
# model. See the note below.
per_position = (predicted_ids == target_ids).mean(axis=0)

plt.figure(figsize=(6, 3))
plt.bar(range(len(per_position)), per_position)
plt.xlabel('position in the sequence'); plt.ylabel('accuracy')
plt.ylim(0, 1.05); plt.title('accuracy by timestep')
plt.tight_layout(); plt.show()

print('accuracy per position:', np.round(per_position, 3).tolist())

## Continuing a sequence the model has never seen

Feeding the model's own prediction back in turns it into a generator — the
same autoregressive loop the language model uses in eval8, on a much simpler
alphabet.


In [ ]:
def continue_sequence(model, start, n_steps=6, seq_len=10, n_col=20):
    """Seed with one number and let the RNN carry on."""
    history = [start]
    for _ in range(n_steps):
        window = (history + [0] * seq_len)[:seq_len]
        x = to_categorical(np.array(window), n_col=n_col)[None, :, :]
        step = min(len(history), seq_len) - 1
        history.append(int(model.predict(x)[0, step].argmax()))
    return history

for start in (0, 3, 7):
    print(f'seeded with {start}: {continue_sequence(model, start)}')

## A caveat: this task does not actually need memory

The model scores 1.000 at every position, including position 0. That is a clue.

The target at step *t* is just `input[t] + 1` — a function of the CURRENT input
alone, with no reference to anything earlier. So a fixed position-wise map
would solve it perfectly too:


In [ ]:
# A hand-built shift matrix: no recurrence, no training, no hidden state.
shift = np.zeros((20, 20))
for value in range(19):
    shift[value, value + 1] = 1.0

print('a fixed position-wise shift scores:',
      round(multiclass_accuracy(test.y, test.X @ shift), 4))
print('the trained RNN scores:           ',
      round(multiclass_accuracy(test.y, predictions), 4))

Both perfect. The counting task validates that the RNN's forward and backward
passes work, but it does not exercise the thing an RNN is *for*. To see the
hidden state earn its place, the target has to depend on something the model
can only know by remembering.

## A task that does need memory

Echo the FIRST element of the sequence at every position. The input at step *t*
says nothing about it once *t* > 0, so the only way through is to carry it
forward in the hidden state.


In [ ]:
def gen_recall(nums, seq_len=8, n_col=10, seed=0):
    """Inputs are random; the target at EVERY step is the first input."""
    rng = np.random.RandomState(seed)
    ids = rng.randint(0, n_col, (nums, seq_len))
    X = np.eye(n_col)[ids]
    y = np.eye(n_col)[np.repeat(ids[:, :1], seq_len, axis=1)]
    return Dataset(X, y)

recall = gen_recall(2000, seed=1)
recall_train, recall_test = train_test_split(recall, split=0.8)

print('input ', recall.X[0].argmax(axis=-1).tolist())
print('target', recall.y[0].argmax(axis=-1).tolist())
print()
# The best a memoryless model can do: repeat the current input.
print('predicting the current input scores:',
      round(multiclass_accuracy(recall.y, recall.X), 4), '<- near chance')

In [ ]:
np.random.seed(2)
recall_model = NN(optimizer=Adam(0.01), epochs=120, verbose=True, step=30)
recall_model.add(RNN(24, bptt_trunc=8, input_shape=(8, 10)))
recall_model.add(SoftMax())
print(recall_model)
recall_model.fit(recall_train)

In [ ]:
recall_pred = recall_model.predict(recall_test.X)
print('test accuracy:', round(multiclass_accuracy(recall_test.y, recall_pred), 4))
print()
for i in range(4):
    print('input    ', recall_test.X[i].argmax(axis=-1).tolist())
    print('predicted', recall_pred[i].argmax(axis=-1).tolist())
    print('target   ', recall_test.y[i].argmax(axis=-1).tolist())
    print()

That one could not have been solved without the hidden state, and it is where
the RNN's two real limits show up:

1. **Everything flows through one hidden state, a step at a time.** A dependency
   between distant positions must survive every intermediate step, which is
   where vanishing gradients bite.
2. **`bptt_trunc` caps how far back the gradient travels.** Nothing beyond that
   window can be learned at all.

Attention removes both: every position reaches every other in a single step,
with no state to squeeze through. That is eval8.

Try next:

1. Lengthen the recall sequence to 20 with `bptt_trunc=5` and watch it fail --
   the dependency is now longer than the gradient window.
2. Echo the first element only at the LAST position, a harder variant still.
